# ARC-AGI — Inspect Failing Tasks

Loads the error analysis JSON produced by `evaluate.py --analyze`, filters to the
worst-performing tasks, and shows for each:

1. **All training pairs** — the rule demonstrations the model was given as context
2. **Test pair comparison** — Test Input | Model Prediction | Actual Answer

Wrong cells in the prediction are highlighted with a red border.

**Cell order:** 1 → 2 → 3 → 4 → 5 → 6

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE        = '/content/drive/MyDrive/arc-agi'
DRIVE_CKPT_DIR    = f'{DRIVE_BASE}/checkpoints'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE}/results'

print('Available error analysis JSONs:')
for p in sorted(Path(DRIVE_RESULTS_DIR).glob('error_analysis_*.json')):
    print(f'  {p.name}  ({p.stat().st_size/1e3:.0f} KB)')

In [ ]:
# ── Cell 3: Clone repo and download ARC training data ───────────────────────
import os, io, urllib.request, zipfile
from pathlib import Path

REPO        = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
if not os.path.isdir(REPO):
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git')
else:
    os.system(f'git -C {REPO} pull --ff-only')
os.chdir(REPO)
print(f'Working directory: {os.getcwd()}')
os.system('git log --oneline -3')

# ARC training data
DATA_DIR = Path('data')
if (DATA_DIR / 'training').exists() and len(list((DATA_DIR / 'training').glob('*.json'))) >= 400:
    print(f"ARC data present ({len(list((DATA_DIR/'training').glob('*.json')))} tasks).")
else:
    print('Downloading ARC-AGI dataset...')
    with urllib.request.urlopen(
        'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    ) as r:
        raw = r.read()
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        dest = DATA_DIR / 'training'
        dest.mkdir(exist_ok=True)
        for m in zf.namelist():
            if 'data/training/' in m and m.endswith('.json'):
                (dest / Path(m).name).write_bytes(zf.read(m))
    print(f"Downloaded: {len(list((DATA_DIR/'training').glob('*.json')))} tasks")

In [ ]:
# ── Cell 4: Config ──────────────────────────────────────────────────────────
RUN_NAME          = 'all_400_arc'   # matches the training run name
K_CONTEXT         = 3               # context pairs fed to the model
SEVERITY_FILTER   = ['total', 'far']  # which severity buckets to inspect
MAX_TASKS         = 30              # max failing tasks to visualise (set 0 for all)

from pathlib import Path

ckpt_path = str(Path(DRIVE_CKPT_DIR) / f'transformer_c{RUN_NAME}_best.pt')
json_path = str(Path(DRIVE_RESULTS_DIR) / f'error_analysis_greedy_transformer_c{RUN_NAME}_best.json')

print(f'Checkpoint : {ckpt_path}')
print(f'  exists   : {Path(ckpt_path).exists()}')
print(f'Error JSON : {json_path}')
print(f'  exists   : {Path(json_path).exists()}')

In [ ]:
# ── Cell 5: Load model + error JSON ─────────────────────────────────────────
import json, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches

sys.path.insert(0, '.')
from src.arc_tokenizer import ArcTokenizer, VOCAB_SIZE
from src.transformer_model import ArcTransformer
from scripts.evaluate import load_checkpoint, greedy_decode

%matplotlib inline

# Colour map
ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
CMAP = mcolors.ListedColormap(ARC_COLORS)
NORM = mcolors.BoundaryNorm(range(11), CMAP.N)

def show_grid(ax, grid, title='', wrong_mask=None):
    """Render a single ARC grid. Wrong cells get a red rectangle border."""
    g = np.array(grid, dtype=np.uint8)
    ax.imshow(g, cmap=CMAP, norm=NORM, interpolation='nearest')
    H, W = g.shape
    ax.set_xticks(np.arange(-0.5, W, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, H, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=0.5)
    ax.tick_params(which='both', bottom=False, left=False,
                   labelbottom=False, labelleft=False)
    if title:
        ax.set_title(title, fontsize=8)
    if wrong_mask is not None:
        for r in range(H):
            for c in range(W):
                if wrong_mask[r, c]:
                    rect = patches.Rectangle(
                        (c - 0.5, r - 0.5), 1, 1,
                        linewidth=2, edgecolor='red', facecolor='red', alpha=0.4
                    )
                    ax.add_patch(rect)

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, saved_args, ckpt_task_ids = load_checkpoint(ckpt_path, device)
tok = ArcTokenizer()

# Load error analysis JSON
with open(json_path) as f:
    analysis = json.load(f)

# Filter to failing tasks (sorted worst first — already sorted in JSON)
failing = [t for t in analysis['tasks'] if t['severity'] in SEVERITY_FILTER]
if MAX_TASKS > 0:
    failing = failing[:MAX_TASKS]

print(f'\nOverall summary:')
for sev, cnt in analysis['summary']['severity_counts'].items():
    pct = analysis['summary']['severity_pct'][sev]
    print(f'  {sev:<8} {cnt:>4} tasks  ({pct:.0f}%)')

print(f'\nShowing {len(failing)} tasks with severity in {SEVERITY_FILTER}:')
print(f'  {"Task ID":<16} {"CellAcc":>7}  {"LOO Exact":>9}  {"Severity":<8}  {"AccStd":>6}')
print(f'  {"-"*16} {"-"*7}  {"-"*9}  {"-"*8}  {"-"*6}')
for t in failing:
    print(f'  {t["task_id"]:<16} {t["cell_acc"]:>7.3f}  '
          f'{t["n_exact"]:>2}/{t["n_pairs"]:<6}  {t["severity"]:<8}  '
          f'{t.get("cell_acc_std", 0):>6.3f}')

In [ ]:
# ── Cell 6: Visualise failing tasks ─────────────────────────────────────────
# For each task:
#   Figure 1 — All training pairs (colour + numbers)
#   Figure 2 — LOO predictions: for each training pair held out, what does
#               the current model predict vs the actual output?
#   Figure 3 — Test pair: Test Input | Model Prediction | Actual Answer
#
# Red shading = wrong cell.  Numbers printed below each figure.
# NOTE: LOO scores in the header come from the error analysis JSON (older
# checkpoint).  The predictions shown here use the CURRENT best checkpoint.

DATA_DIR = Path('data/training')


def print_grids_side_by_side(grids_and_labels, separator='   '):
    arrays    = [np.array(g, dtype=np.uint8) for g, _ in grids_and_labels]
    labels    = [lbl for _, lbl in grids_and_labels]
    rows_list = [[' '.join(f'{v:1d}' for v in row) for row in a] for a in arrays]
    n_rows    = max(len(r) for r in rows_list)
    widths    = [max(len(rows[0]) if rows else 0, len(lbl))
                 for rows, lbl in zip(rows_list, labels)]
    for i, rows in enumerate(rows_list):
        while len(rows) < n_rows:
            rows.append('')
    print(separator.join(lbl.ljust(widths[i]) for i, lbl in enumerate(labels)))
    print(separator.join('-' * w for w in widths))
    for row_idx in range(n_rows):
        print(separator.join(
            rows_list[i][row_idx].ljust(widths[i]) for i in range(len(arrays))
        ))
    print()


def print_diff(pred, target):
    wrong   = pred != target
    n_wrong = int(wrong.sum())
    H, W    = target.shape
    print(f'Diff ({n_wrong}/{target.size} cells wrong)  . = correct, digit = predicted value:')
    for r in range(H):
        print(' '.join('.' if not wrong[r, c] else str(pred[r, c]) for c in range(W)))
    print()


for task_info in failing:
    tid  = task_info['task_id']
    path = DATA_DIR / f'{tid}.json'
    if not path.exists():
        print(f'Missing data file: {tid}')
        continue

    task       = json.loads(path.read_text())
    task['task_id'] = tid
    train      = task['train']
    test_pairs = task.get('test', [])
    n_train    = len(train)

    header = (f'Task {tid}  |  JSON LOO cell_acc={task_info["cell_acc"]:.3f}  '
              f'severity={task_info["severity"]}  '
              f'({task_info["n_exact"]}/{task_info["n_pairs"]} LOO exact  '
              f'— from older checkpoint)')
    print('=' * 70)
    print(header)
    print('=' * 70)

    # ── Figure 1: Training pairs ──────────────────────────────────────────────
    fig1, ax1 = plt.subplots(2, n_train, figsize=(3 * n_train, 6), squeeze=False)
    fig1.suptitle(f'Task {tid} — Training pairs', fontsize=9, fontweight='bold')
    for i, pair in enumerate(train):
        ctx_note = f'(ctx {i+1})' if i < K_CONTEXT else f'(pair {i+1}, not in ctx)'
        show_grid(ax1[0, i], pair['input'],  title=f'Train {i+1} Input\n{ctx_note}')
        show_grid(ax1[1, i], pair['output'], title=f'Train {i+1} Output')
    fig1.tight_layout()
    plt.show()

    print('Training pairs (numbers):')
    for i, pair in enumerate(train):
        ctx_note = 'context' if i < K_CONTEXT else 'not in context'
        print_grids_side_by_side([
            (np.array(pair['input'],  dtype=np.uint8), f'Train {i+1} Input ({ctx_note})'),
            (np.array(pair['output'], dtype=np.uint8), f'Train {i+1} Output'),
        ])

    # ── Figure 2: LOO predictions (current checkpoint) ───────────────────────
    print(f'\nLOO predictions (current checkpoint):')
    loo_results = []
    for hi in range(n_train):
        ctx_raw = [p for i, p in enumerate(train) if i != hi]
        tp      = train[hi]
        test_in = np.array(tp['input'],  dtype=np.uint8)
        target  = np.array(tp['output'], dtype=np.uint8)
        H, W    = target.shape
        ctx     = [(np.array(p['input'],  dtype=np.uint8),
                    np.array(p['output'], dtype=np.uint8))
                   for p in ctx_raw[:K_CONTEXT]]
        pred    = greedy_decode(model, tok, ctx, test_in, H, W, device)
        ca      = float((pred == target).mean())
        wrong   = pred != target
        loo_results.append((hi, test_in, pred, target, ca, wrong))

    n_loo   = len(loo_results)
    fig2, ax2 = plt.subplots(3, n_loo, figsize=(3 * n_loo, 8), squeeze=False)
    fig2.suptitle(f'Task {tid} — LOO predictions (current checkpoint)',
                  fontsize=9, fontweight='bold')
    for col, (hi, test_in, pred, target, ca, wrong) in enumerate(loo_results):
        n_wrong = int(wrong.sum())
        show_grid(ax2[0, col], test_in, title=f'Hold out {hi+1}\nInput')
        show_grid(ax2[1, col], pred,    title=f'Prediction\nacc={ca:.2f}',
                  wrong_mask=wrong if n_wrong > 0 else None)
        show_grid(ax2[2, col], target,  title='Actual')
    fig2.tight_layout()
    plt.show()

    for hi, test_in, pred, target, ca, wrong in loo_results:
        n_ctx = n_train - 1
        print(f'Hold out Train {hi+1}  (context = other {min(n_ctx, K_CONTEXT)} pair(s)):')
        print_grids_side_by_side([
            (test_in, f'Train {hi+1} Input'),
            (pred,    'Prediction'),
            (target,  'Actual'),
        ])
        print_diff(pred, target)

    # ── Figure 3: Test pair (current checkpoint) ──────────────────────────────
    ctx_all = [(np.array(p['input'],  dtype=np.uint8),
                np.array(p['output'], dtype=np.uint8))
               for p in train[:K_CONTEXT]]
    test_results = []
    for tp in test_pairs:
        if 'output' not in tp:
            continue
        test_in = np.array(tp['input'],  dtype=np.uint8)
        target  = np.array(tp['output'], dtype=np.uint8)
        H, W    = target.shape
        pred    = greedy_decode(model, tok, ctx_all, test_in, H, W, device)
        ca      = float((pred == target).mean())
        wrong   = pred != target
        test_results.append((test_in, pred, target, ca, wrong))

    for j, (test_in, pred, target, ca, wrong) in enumerate(test_results):
        n_wrong = int(wrong.sum())
        fig3, ax3 = plt.subplots(1, 3, figsize=(9, 4), squeeze=False)
        fig3.suptitle(
            f'Task {tid} — Test pair {j+1}  '
            f'(current checkpoint)  acc={ca:.3f}  ({n_wrong}/{target.size} wrong)',
            fontsize=9, fontweight='bold'
        )
        show_grid(ax3[0, 0], test_in, title='Test Input')
        show_grid(ax3[0, 1], pred,    title='Prediction\n(red = wrong)',
                  wrong_mask=wrong if n_wrong > 0 else None)
        show_grid(ax3[0, 2], target,  title='Actual Answer')
        fig3.tight_layout()
        plt.show()

        print(f'Test pair {j+1} (numbers):')
        print_grids_side_by_side([
            (test_in, 'Test Input'),
            (pred,    'Prediction'),
            (target,  'Actual'),
        ])
        print_diff(pred, target)

    print()
